In [100]:
# ### Fetch Complete Historical Weather via Open-Meteo (2014-2025)
import requests
import pandas as pd
from pathlib import Path

# Setup paths
REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = REPO_ROOT / "data" / "raw" / "weather"
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Fetching continuous historical weather (2014-2025)...")

# Open-Meteo Historical Archive API
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 43.6532,
    "longitude": -79.3832,
    "start_date": "2014-01-01",
    "end_date": "2025-12-31",
    # Requesting the exact variables needed for collision risk modelling
    "daily": ["temperature_2m_mean", "temperature_2m_min", "temperature_2m_max",
              "precipitation_sum", "snowfall_sum", "wind_speed_10m_max"],
    "timezone": "America/Toronto"
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    daily_data = data["daily"]

    # Parse the JSON directly into a clean Pandas DataFrame
    weather_df = pd.DataFrame({
        # FIX: Wrapped in pd.Series() so the .dt accessor works perfectly
        "date": pd.to_datetime(pd.Series(daily_data["time"])).dt.date,
        "tavg": daily_data["temperature_2m_mean"],
        "tmin": daily_data["temperature_2m_min"],
        "tmax": daily_data["temperature_2m_max"],
        "prcp": daily_data["precipitation_sum"], # in mm
        "snow": daily_data["snowfall_sum"],      # in cm
        "wspd": daily_data["wind_speed_10m_max"] # in km/h
    })

    # Save the continuous historical weather
    out_path = DATA_RAW / "toronto_historical_weather_daily.parquet"
    weather_df.to_parquet(out_path, index=False)
    print(f"Success! Saved {len(weather_df)} days of weather data to: {out_path}")

    # Display the first few rows
    display(weather_df.head())
else:
    print(f"API Error {response.status_code}: {response.text}")

Fetching continuous historical weather (2014-2025)...
Success! Saved 4383 days of weather data to: C:\code\pyspark-playground\Covercheck-Toronto\data\raw\weather\toronto_historical_weather_daily.parquet


,date,tavg,tmin,tmax,prcp,snow,wspd
0,2014-01-01,-11.1,-14.8,-8.8,0.0,0.21,18.0
1,2014-01-02,-17.3,-19.3,-14.9,2.5,2.24,23.1
2,2014-01-03,-17.8,-22.7,-10.9,0.0,0.00,20.3
3,2014-01-04,-4.6,-9.8,-1.2,0.2,0.35,28.2
4,2014-01-05,-0.6,-1.8,2.0,12.2,7.35,20.6
